<a href="https://colab.research.google.com/github/sachinn854/Brain-Tumor-Segmentatiton/blob/main/notebooks/train_brats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WAS-Mamba on BraTS -- Real Training

Run `notebooks/smoke_test.ipynb` first if you haven't -- this notebook assumes the base model, loss, and data loader are already confirmed working.

**Design so far:**
- **Data** goes to `/content/` (Colab's local disk) -- re-downloaded fresh each session. The full BraTS2021 archive is ~13.4GB; Colab's local disk has plenty of room, and Kaggle-to-Colab download is fast. This deliberately avoids using Google Drive's free 15GB quota for something re-downloadable.
- **Checkpoints** go to **Google Drive** (mounted below) -- small, and must survive across Colab sessions since training (1000 epochs, per the paper) will not fit in one ~12h session. Training auto-resumes from the latest checkpoint on Drive every time this notebook is re-run.

Before running: **Runtime -> Change runtime type -> T4 GPU** (or better, if you have Colab Pro).

## 1. Mount Drive (for checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = '/content/drive/MyDrive/wasmamba_checkpoints'
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {CHECKPOINT_DIR}')

## 2. Get the code

In [ ]:
import os

REPO_URL = "github.com/sachinn854/Brain-Tumor-Segmentatiton.git"
REPO_DIR = "Brain-Tumor-Segmentatiton"

if not os.path.isdir(f"/content/{REPO_DIR}"):
    get_ipython().system(f'git clone https://{REPO_URL}')
else:
    get_ipython().system(f'git -C /content/{REPO_DIR} pull')

get_ipython().run_line_magic('cd', f'/content/{REPO_DIR}')

## 3. Install dependencies

Same as the smoke test notebook -- see that one for why each flag is needed.

In [ ]:
import torch

major, minor = torch.cuda.get_device_capability(0)
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{major}.{minor}"
print(f"Building only for compute capability {major}.{minor}")

!pip install -q einops timm nibabel ninja packaging kaggle
!pip install -q causal-conv1d --no-build-isolation
!pip install -q mamba-ssm --no-build-isolation

## 4. Download the full BraTS2021 dataset (~13.4GB)

Needs a `KAGGLE_API_TOKEN` Colab Secret (see smoke_test.ipynb section 8 for how to add one -- it's account-linked, so if you already added it there, it's available here too).

This is the full 1251-case archive, not the ~10MB single-case test file from the smoke test. Expect several minutes.

In [ ]:
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

DATA_ROOT = '/content/BraTS2021'

if not os.path.isdir(DATA_ROOT):
    !kaggle datasets download -d dschettler8845/brats-2021-task1 -f BraTS2021_Training_Data.tar
    !mkdir -p /content/brats_raw_extract
    !tar -xf BraTS2021_Training_Data.tar -C /content/brats_raw_extract
    print('Extracted. Organizing into <case_id>/<files> layout next.')
else:
    print(f'{DATA_ROOT} already exists, skipping download.')

## 5. Organize into the layout `BratsDataset` expects

Regardless of whether the extracted archive already has one folder per case or dumps everything flat (the single-case test .tar in the smoke test was flat -- this full archive's internal layout wasn't verified in advance), this scans for every `*_flair.nii.gz` / `*_seg.nii.gz` etc. under the extraction folder and sorts them into `DATA_ROOT/<case_id>/<file>` by the case-ID prefix in each filename. Safe to re-run.

In [ ]:
import glob
import re
import shutil

if not os.path.isdir(DATA_ROOT):
    os.makedirs(DATA_ROOT, exist_ok=True)
    all_files = glob.glob('/content/brats_raw_extract/**/*.nii.gz', recursive=True)
    print(f'Found {len(all_files)} .nii.gz files to organize')

    moved, skipped = 0, 0
    for fpath in all_files:
        fname = os.path.basename(fpath)
        match = re.match(r'(BraTS2021_\d+)_', fname)
        if not match:
            skipped += 1
            continue
        case_id = match.group(1)
        case_dir = os.path.join(DATA_ROOT, case_id)
        os.makedirs(case_dir, exist_ok=True)
        dest = os.path.join(case_dir, fname)
        if not os.path.exists(dest):
            shutil.move(fpath, dest)
            moved += 1

    print(f'Moved {moved} files, skipped {skipped} (unrecognized naming)')
    shutil.rmtree('/content/brats_raw_extract', ignore_errors=True)

n_cases = len([d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))])
print(f'{n_cases} cases ready in {DATA_ROOT}')

## 6. Smoke-test the training loop itself (5 epochs)

**Retry note:** the first attempt at this OOM'd on the T4 at the default batch_size=2. Since then, `src/configs/wasmamba_config.py` defaults changed to `batch_size=1` + gradient checkpointing ON (both wired through the architecture) specifically to fit a 16GB card -- that fix was never actually tested on Colab before switching to local training, so this is worth one more try before ruling Colab out for good.

Before committing to the paper's real 1000 epochs, confirm the training loop -- optimizer step, checkpoint save, checkpoint resume, validation, Dice computation -- all actually work, and check whether it OOMs again. This should take a few minutes, not hours.

In [ ]:
!git -C /content/Brain-Tumor-Segmentatiton pull

!python -m src.engine.train \
    --data_path {DATA_ROOT} \
    --checkpoint_dir {CHECKPOINT_DIR} \
    --epochs 5

If that ran cleanly and printed per-epoch train/val loss and per-class Dice with no errors: the full loop works, and Colab is viable after all. Delete the checkpoint before starting the real run, since --epochs 5 wrote a checkpoint that the next run would otherwise "resume" from (thinking training is already 5 epochs in).

In [ ]:
# Only run this if the 5-epoch smoke test above succeeded and you're about
# to start the REAL run -- this deletes the smoke-test checkpoint so the
# real run starts fresh from epoch 1, not "resuming" from epoch 5.
smoke_test_ckpt = os.path.join(CHECKPOINT_DIR, 'latest.pth')
if os.path.exists(smoke_test_ckpt):
    os.remove(smoke_test_ckpt)
    print('Removed smoke-test checkpoint. Ready for a real run.')
else:
    print('No checkpoint found -- nothing to remove.')

## 7. Real training (1000 epochs, per the paper)

This WILL exceed a single Colab session. That's fine -- it auto-saves a checkpoint every epoch to Drive, and re-running this exact cell after a disconnect (Runtime -> re-run, or a fresh session with Drive re-mounted) picks up from the last completed epoch automatically. Just keep re-running this cell across sessions until `epoch` reaches `cfg.epochs`.

In [ ]:
!python -m src.engine.train \
    --data_path {DATA_ROOT} \
    --checkpoint_dir {CHECKPOINT_DIR}